# Spark ML Class Demo
**Dataset:** NYC Yellow Taxi Trip Records — Jan 2023 (~3M rows)

This notebook follows the lecture slides and demonstrates every major Spark ML concept:

| Slide | Topic | Demo |
|-------|-------|------|
| 1 | Basic Statistics | Correlation, Summarizer, ChiSquareTest |
| 2–3 | ML Pipeline Concepts | DataFrame, Transformer, Estimator, Pipeline |
| 4 | Pipeline as DAG | Linear pipeline: VectorAssembler → Scaler → LR |
| 5 | Parameters & Persistence | ParamMap, save/load pipeline |
| 6 | Model Selection & Tuning | CrossValidator + ParamGridBuilder |

---

## 0. Setup

In [ ]:
import os
import shutil

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# ML: features
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer

# ML: algorithms
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans

# ML: pipeline & tuning
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import (
    RegressionEvaluator,
    BinaryClassificationEvaluator,
    ClusteringEvaluator,
    MulticlassClassificationEvaluator,
)

# ML: statistics
from pyspark.ml.stat import Correlation, ChiSquareTest, Summarizer
from pyspark.ml.linalg import Vectors

print("All imports OK")

In [ ]:
spark = (
    SparkSession.builder
    .appName("SparkML_Taxi_Demo")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version} | {spark.sparkContext.defaultParallelism} workers")

# Path to taxi data
DATA_DIR  = "/home/alka/MLOPs-Class/spark_demo"
DATA_2023 = os.path.join(DATA_DIR, "yellow_tripdata_2023-01.parquet")
MODEL_DIR = "/tmp/sparkml_demo_model"

---
## 1. Load & Prepare Data

We load the Jan 2023 taxi data, engineer features, and cache a clean sample for all ML tasks.

In [ ]:
raw = spark.read.parquet(DATA_2023)
print(f"Raw rows: {raw.count():,}")
raw.printSchema()

In [ ]:
# Feature engineering: add derived columns
df = (
    raw
    .withColumn(
        "duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0
    )
    .withColumn("hour_of_day",  F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week",  F.dayofweek("tpep_pickup_datetime"))
    # Clean: sensible ranges only
    .filter(
        (F.col("fare_amount")    >  0)   & (F.col("fare_amount")    < 200) &
        (F.col("trip_distance")  >  0)   & (F.col("trip_distance")  <  50) &
        (F.col("duration_min")   >  0)   & (F.col("duration_min")   < 120) &
        (F.col("tip_amount")     >= 0)   & (F.col("tip_amount")     <  60) &
        (F.col("passenger_count").isNotNull())
    )
    .select(
        "VendorID", "passenger_count", "trip_distance",
        "PULocationID", "DOLocationID", "payment_type",
        "fare_amount", "tip_amount", "total_amount",
        "duration_min", "hour_of_day", "day_of_week",
    )
)

# Sample 5% rows for speed up
df_sample = df.sample(fraction=0.05, seed=42).cache()

n = df_sample.count()
print(f"Clean sample: {n:,} rows")
df_sample.show(5)

---
## 2. Basic Statistics 

> Spark ML provides: **Correlation** (Pearson / Spearman), **ChiSquareTest** (independence for categorical features), and **Summarizer** (column-wise max, min, mean, variance, std, count of non-zeros).

### 2.1 Correlation Matrix (Pearson)

We first assemble numeric features into a **vector column** — required by all Spark ML statistics APIs.

In [ ]:
# Step 1: assemble numeric columns into a single vector column
stat_cols = ["trip_distance", "fare_amount", "tip_amount", "duration_min", "passenger_count"]

assembler_stat = VectorAssembler(inputCols=stat_cols, outputCol="stat_features")
df_vec = assembler_stat.transform(df_sample).select("stat_features")

# Show the first row of the vectorized features
df_vec.show(1, truncate=False)

# Step 2: compute Pearson correlation matrix
corr_matrix = Correlation.corr(df_vec, "stat_features", method="pearson")
corr_array  = corr_matrix.collect()[0][0].toArray()
# display DAG of correlation computation
print("\nDAG of correlation computation:")
corr_matrix.explain(True)

# Display as a Pandas DataFrame for readability
corr_df = pd.DataFrame(corr_array, index=stat_cols, columns=stat_cols).round(3)
print("Pearson Correlation Matrix:")
print(corr_df)

# Heatmap
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr_array, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(stat_cols))); ax.set_xticklabels(stat_cols, rotation=30, ha="right")
ax.set_yticks(range(len(stat_cols))); ax.set_yticklabels(stat_cols)
for i in range(len(stat_cols)):
    for j in range(len(stat_cols)):
        ax.text(j, i, f"{corr_array[i,j]:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax)
ax.set_title("Pearson Correlation — NYC Taxi Features")
plt.tight_layout()
plt.show()

### 2.1b Spearman Correlation

Spearman uses rank-based correlation — more robust to outliers.

In [ ]:
corr_spearman = Correlation.corr(df_vec, "stat_features", method="spearman")
spearman_array = corr_spearman.collect()[0][0].toArray()
print("Spearman Correlation (trip_distance vs fare_amount):",
      round(spearman_array[0, 1], 4))
print("Pearson  Correlation (trip_distance vs fare_amount):",
      round(corr_array[0, 1], 4))

### 2.2 Summarizer

> Available metrics: **max, min, mean, sum, variance, std, numNonZeros, count**.

In [ ]:
# Summarizer computes column-wise stats on a vector column
summary = df_vec.select(
    Summarizer.metrics("mean", "max", "min", "variance", "std")
              .summary(F.col("stat_features"))
)
summary.show(truncate=False)

# Extract individual metric
means = summary.collect()[0][0]["mean"].toArray()
print("\nColumn means:")
for col_name, val in zip(stat_cols, means):
    print(f"  {col_name:<20}: {val:.4f}")

### 2.3 Chi-Square Test (Hypothesis Testing)

> `ChiSquareTest` runs Pearson's **independence test** for every feature against the label.  
> Both features and label **must be categorical** (double-typed).  
> A low p-value means the feature and label are **not independent** — the feature is informative.

In [ ]:
# Use VendorID and day_of_week as categorical features; payment_type as label
chi_cols  = ["VendorID", "day_of_week", "hour_of_day", "passenger_count"]
chi_label = "payment_type"

assembler_chi = VectorAssembler(
    inputCols=chi_cols, outputCol="chi_features"
)
df_chi = (
    assembler_chi.transform(df_sample)
    .select("chi_features", F.col(chi_label).cast("double").alias("label"))
    .filter(F.col("label").isin([1.0, 2.0]))  # credit=1, cash=2
)

chi_result = ChiSquareTest.test(df_chi, "chi_features", "label")

p_values  = chi_result.select("pValues").collect()[0][0].toArray()
stats     = chi_result.select("statistics").collect()[0][0].toArray()

print(f"{'Feature':<20} {'Chi² stat':>12} {'p-value':>12} {'Independent?':>14}")
print("-" * 62)
for feat, stat, pval in zip(chi_cols, stats, p_values):
    flag = "NO  (informative)" if pval < 0.05 else "YES (not useful)"
    print(f"  {feat:<18} {stat:>12.2f} {pval:>12.4f} {flag:>14}")

---
## 3. ML Pipeline Concepts  

The five main abstractions:

| Concept | Role | Taxi Example |
|---------|------|--------------|
| **DataFrame** | ML dataset | taxi trips with columns for distance, fare, features, label |
| **Transformer** | `DataFrame → DataFrame` | `VectorAssembler` appends a `features` column |
| **Estimator** | `fit(DataFrame) → Transformer` | `StandardScaler.fit(df)` learns means/stds → `StandardScalerModel` |
| **Pipeline** | chains Transformers + Estimators | `[Assembler, Scaler, LR]` |
| **Parameter** | uniform API | `lr.setMaxIter(10)` or `ParamMap(lr.maxIter→10)` |

### 3.1 DataFrame — the ML dataset

In [ ]:
# The DataFrame holds raw features, engineered features, labels, and predictions side-by-side
print("DataFrame columns:", df_sample.columns)
print(f"Rows: {df_sample.count():,}")
df_sample.select("trip_distance", "fare_amount", "tip_amount", "payment_type").show(5)

### 3.2 Transformer — `VectorAssembler`

A Transformer has a **`transform(df)`** method.  
It **appends** one or more columns — it never modifies existing columns.  
Each instance has a **unique UID**.

In [ ]:
feature_cols = ["trip_distance", "fare_amount", "passenger_count",
                "duration_min", "hour_of_day", "day_of_week",
                "PULocationID", "DOLocationID"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Each Transformer has a unique id
print(f"Transformer uid  : {assembler.uid}")
print(f"Type             : {type(assembler).__name__}")

# transform() appends the 'features' column — returns a NEW DataFrame
df_assembled = assembler.transform(df_sample)
print(f"\nColumns before: {len(df_sample.columns)}")
print(f"Columns after : {len(df_assembled.columns)}  (added: 'features')")
df_assembled.select("trip_distance", "fare_amount", "features").show(3, truncate=60)

### 3.3 Estimator — `StandardScaler`

An Estimator has a **`fit(df)`** method that **learns** from the data  
and returns a trained **Transformer** (a `StandardScalerModel`).  

> `Transformer.transform()` and `Estimator.fit()` are **stateless** — the  
> input DataFrame is never modified; a new one is returned.

In [ ]:
scaler = StandardScaler(
    inputCol="features", outputCol="scaled_features",
    withMean=True, withStd=True
)

print(f"Estimator uid    : {scaler.uid}")
print(f"Type             : {type(scaler).__name__}  ← Estimator (not yet trained)")

# fit() learns mean/std from the TRAINING data → produces a StandardScalerModel
scaler_model = scaler.fit(df_assembled)
print(f"\nAfter fit()")
print(f"Model type       : {type(scaler_model).__name__}  ← Transformer (trained)")
print(f"Learned means    : {scaler_model.mean.toArray().round(3)}")

# The model (Transformer) can now transform new data
df_scaled = scaler_model.transform(df_assembled)
df_scaled.select("features", "scaled_features").show(2, truncate=70)

### 3.4 Pipeline — chaining stages

A `Pipeline` is itself an **Estimator**.  
`pipeline.fit(df)` calls each stage in order:  
- If a stage is an **Estimator**, `fit()` is called and the resulting model is used for `transform()`  
- If a stage is a **Transformer**, `transform()` is called directly

In [ ]:
# Build a minimal demo pipeline: Assembler → Scaler
demo_assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
demo_scaler    = StandardScaler(inputCol="raw_features", outputCol="scaled_features",
                                withMean=True, withStd=True)

demo_pipeline = Pipeline(stages=[demo_assembler, demo_scaler])

print("Pipeline stages:")
for i, stage in enumerate(demo_pipeline.getStages()):
    print(f"  [{i}] {type(stage).__name__:<25}  uid={stage.uid}")

# fit() trains the Pipeline (StandardScaler learns stats; VectorAssembler has nothing to learn)
demo_model = demo_pipeline.fit(df_sample)
print(f"\nAfter fit(): type = {type(demo_model).__name__}")

# transform() runs all stages in sequence
df_out = demo_model.transform(df_sample)
print(f"Output columns: {[c for c in df_out.columns if c not in df_sample.columns]}")

---
## 4. Regression Pipeline — Predict Tip Amount 

**Task:** Given trip features, predict `tip_amount`.

**Linear Pipeline:**
```
df  →  VectorAssembler  →  StandardScaler  →  LinearRegression  →  predictions
       (Transformer)       (Estimator)         (Estimator)
```

Only credit-card trips (`payment_type == 1`) have non-zero tips.

In [ ]:
# Keep only credit-card trips (payment_type=1) where tips are meaningful
df_cc = df_sample.filter(F.col("payment_type") == 1)

train_reg, test_reg = df_cc.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_reg.count():,}  |  Test: {test_reg.count():,}")

# Show tip distribution
df_cc.select("tip_amount").describe().show()

In [ ]:
reg_features = ["trip_distance", "fare_amount", "passenger_count",
                "duration_min", "hour_of_day", "day_of_week"]

# Stage 1: Transformer — assemble feature columns into a vector
reg_assembler = VectorAssembler(
    inputCols=reg_features, outputCol="raw_features"
)

# Stage 2: Estimator — scale features (learns mean/std on training data)
reg_scaler = StandardScaler(
    inputCol="raw_features", outputCol="features",
    withMean=True, withStd=True
)

# Stage 3: Estimator — fit a linear regression model
lr = LinearRegression(
    featuresCol="features", labelCol="tip_amount",
    maxIter=20, regParam=0.1
)

reg_pipeline = Pipeline(stages=[reg_assembler, reg_scaler, lr])

print("Regression Pipeline stages:")
for i, s in enumerate(reg_pipeline.getStages()):
    role = "Transformer" if hasattr(s, 'transform') and not hasattr(s, 'fit') else "Estimator"
    print(f"  [{i}] {type(s).__name__:<25}")

# Train the entire pipeline on training data
reg_model = reg_pipeline.fit(train_reg)
print("\nPipeline fitted successfully.")

In [ ]:
# Predict on test set
predictions_reg = reg_model.transform(test_reg)
predictions_reg.select("tip_amount", "prediction", "trip_distance", "fare_amount").show(8)

evaluator_rmse = RegressionEvaluator(labelCol="tip_amount", metricName="rmse")
evaluator_r2   = RegressionEvaluator(labelCol="tip_amount", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions_reg)
r2   = evaluator_r2.evaluate(predictions_reg)

print(f"\nTest RMSE : {rmse:.4f}")
print(f"Test R²   : {r2:.4f}")

# Extract the trained LR model from the pipeline
lr_model = reg_model.stages[-1]
print(f"\nLR Coefficients (top 3): {lr_model.coefficients.toArray()[:3].round(4)}")
print(f"LR Intercept           : {lr_model.intercept:.4f}")

In [ ]:
# Scatter: actual vs predicted tips
preds_pd = predictions_reg.select("tip_amount", "prediction").limit(2000).toPandas()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(preds_pd["tip_amount"], preds_pd["prediction"],
           alpha=0.3, s=10, color="steelblue")
ax.plot([0, 20], [0, 20], "r--", label="perfect prediction")
ax.set_xlabel("Actual tip ($)")
ax.set_ylabel("Predicted tip ($)")
ax.set_title(f"Regression: Actual vs Predicted Tip\nRMSE={rmse:.3f}  R²={r2:.3f}")
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Classification Pipeline — High Tip Prediction  

**Task:** Predict whether a credit-card trip will have a **high tip** (tip > 20% of fare).

```
df  →  VectorAssembler  →  LogisticRegression  →  predictions (label: 0/1)
```

In [ ]:
# Create binary label: high_tip = 1 if tip > 20% of fare
df_clf = df_cc.withColumn(
    "high_tip",
    (F.col("tip_amount") / F.col("fare_amount") > 0.20).cast("double")
)

print("Label distribution:")
df_clf.groupBy("high_tip").count().orderBy("high_tip").show()

train_clf, test_clf = df_clf.randomSplit([0.8, 0.2], seed=42)

In [ ]:
clf_features = ["trip_distance", "fare_amount", "passenger_count",
                "duration_min", "hour_of_day", "day_of_week"]

clf_assembler = VectorAssembler(inputCols=clf_features, outputCol="features")

logreg = LogisticRegression(
    featuresCol="features", labelCol="high_tip",
    maxIter=20, regParam=0.01
)

clf_pipeline = Pipeline(stages=[clf_assembler, logreg])

clf_model = clf_pipeline.fit(train_clf)
predictions_clf = clf_model.transform(test_clf)

# Evaluate
auc = BinaryClassificationEvaluator(
    labelCol="high_tip", metricName="areaUnderROC"
).evaluate(predictions_clf)

acc = MulticlassClassificationEvaluator(
    labelCol="high_tip", predictionCol="prediction", metricName="accuracy"
).evaluate(predictions_clf)

print(f"AUC-ROC  : {auc:.4f}")
print(f"Accuracy : {acc:.4f}")

predictions_clf.select("high_tip", "prediction", "probability").show(6)

---
## 6. Clustering — Discover Trip Patterns  

**Task:** Group taxi trips into natural clusters using **K-Means**.

```
df  →  VectorAssembler  →  StandardScaler  →  KMeans  →  cluster assignments
```

In [ ]:
cluster_features = ["trip_distance", "fare_amount", "duration_min"]

km_assembler = VectorAssembler(inputCols=cluster_features, outputCol="raw_features")
km_scaler    = StandardScaler(inputCol="raw_features", outputCol="features",
                               withMean=True, withStd=True)
kmeans       = KMeans(k=3, seed=42, featuresCol="features", predictionCol="cluster")

km_pipeline  = Pipeline(stages=[km_assembler, km_scaler, kmeans])
km_model     = km_pipeline.fit(df_sample)

clusters = km_model.transform(df_sample)

# Cluster stats
print("Cluster sizes:")
clusters.groupBy("cluster").count().orderBy("cluster").show()

print("Cluster centroids (scaled space):")
km_model_stage = km_model.stages[-1]
for i, center in enumerate(km_model_stage.clusterCenters()):
    print(f"  Cluster {i}: {center.round(3)}")

In [ ]:
# Silhouette score (ClusteringEvaluator)
silhouette = ClusteringEvaluator(
    featuresCol="features", predictionCol="cluster"
).evaluate(clusters)
print(f"Silhouette score (k=3): {silhouette:.4f}")

# Cluster profiles in original space
print("\nCluster mean profiles (original scale):")
clusters.groupBy("cluster").agg(
    F.round(F.avg("trip_distance"), 2).alias("avg_dist_mi"),
    F.round(F.avg("fare_amount"),   2).alias("avg_fare_$"),
    F.round(F.avg("duration_min"),  2).alias("avg_dur_min"),
    F.count("*").alias("count")
).orderBy("cluster").show()

In [ ]:
# Visualize clusters: trip_distance vs fare_amount, colored by cluster
plot_df = clusters.select("trip_distance", "fare_amount", "cluster") \
                  .limit(3000).toPandas()

colors = ["#4C72B0", "#DD8452", "#55A868"]
fig, ax = plt.subplots(figsize=(7, 5))
for cid in sorted(plot_df["cluster"].unique()):
    sub = plot_df[plot_df["cluster"] == cid]
    ax.scatter(sub["trip_distance"], sub["fare_amount"],
               c=colors[cid], alpha=0.4, s=10, label=f"Cluster {cid}")
ax.set_xlabel("Trip Distance (mi)")
ax.set_ylabel("Fare Amount ($)")
ax.set_title("K-Means Clustering of NYC Taxi Trips (k=3)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Parameters & Model Tuning 

### 7.1 The Parameters API

> MLlib uses a **uniform parameter API**: parameters can be set directly on an instance  
> (`lr.setMaxIter(10)`) or passed via a `ParamMap` to `fit()`/`transform()`.

In [ ]:
from pyspark.ml.param import Param

lr1 = LogisticRegression(featuresCol="features", labelCol="high_tip")
lr2 = LogisticRegression(featuresCol="features", labelCol="high_tip")

# Method 1: set directly on instance
lr1.setMaxIter(5)
lr1.setRegParam(0.01)

# Method 2: use ParamMap — lets you pass DIFFERENT values for different instances
from pyspark.ml.tuning import ParamGridBuilder
paramMap = {lr1.maxIter: 10, lr2.maxIter: 20}   # dict form of ParamMap

print("lr1 uid :", lr1.uid)
print("lr2 uid :", lr2.uid)
print("lr1 maxIter:", lr1.getMaxIter())
print("lr1 regParam:", lr1.getRegParam())
print("\nParamMap (lr1.maxIter→10, lr2.maxIter→20):")
print("  lr1.maxIter →", paramMap[lr1.maxIter])
print("  lr2.maxIter →", paramMap[lr2.maxIter])
print()
print("Note: same type (LogisticRegression), but DIFFERENT uids → allowed in same Pipeline")

### 7.2 Cross Validator 

Equivalent to **GridSearchCV** in scikit-learn:
1. Split data into **k folds**
2. For each parameter combination: train on k-1 folds, validate on remaining fold
3. Compute **average metric** across folds
4. Choose **best parameters** and retrain on full dataset

Use `cv.setParallelism(n)` to evaluate multiple parameter combos concurrently.

In [ ]:
cv_assembler = VectorAssembler(inputCols=clf_features, outputCol="features")
cv_lr        = LogisticRegression(featuresCol="features", labelCol="high_tip")
cv_pipeline  = Pipeline(stages=[cv_assembler, cv_lr])

# ParamGridBuilder: defines the hyperparameter search grid
param_grid = (
    ParamGridBuilder()
    .addGrid(cv_lr.maxIter,  [10, 30])       # 2 values
    .addGrid(cv_lr.regParam, [0.01, 0.1])    # 2 values  → 4 combos total
    .build()
)

print(f"Total parameter combinations: {len(param_grid)}")
for i, pg in enumerate(param_grid):
    print(f"  Combo {i}: maxIter={pg[cv_lr.maxIter]}  regParam={pg[cv_lr.regParam]}")

In [ ]:
cv_evaluator = BinaryClassificationEvaluator(
    labelCol="high_tip", metricName="areaUnderROC"
)

cv = CrossValidator(
    estimator    = cv_pipeline,
    estimatorParamMaps = param_grid,
    evaluator    = cv_evaluator,
    numFolds     = 3,           # 3-fold CV
    parallelism  = 4,           # evaluate up to 4 combos concurrently
    seed         = 42,
)

print(f"Running {len(param_grid)}-combo × {3}-fold CV (= {len(param_grid)*3} model trains)...")
cv_model = cv.fit(train_clf)
print("Done.")

# Best model AUC on validation
best_auc_cv = max(cv_model.avgMetrics)
best_idx    = cv_model.avgMetrics.index(best_auc_cv)
best_params = param_grid[best_idx]

print(f"\nCV Average AUC per combo:")
for i, (pg, metric) in enumerate(zip(param_grid, cv_model.avgMetrics)):
    marker = " ← best" if i == best_idx else ""
    print(f"  maxIter={pg[cv_lr.maxIter]:>3}  regParam={pg[cv_lr.regParam]:.3f}  AUC={metric:.4f}{marker}")

# Test set performance of best model
best_preds = cv_model.transform(test_clf)
best_auc_test = cv_evaluator.evaluate(best_preds)
print(f"\nBest model Test AUC: {best_auc_test:.4f}")

---
## 8. ML Persistence — Save & Load Pipeline 

In [ ]:
import shutil

MODEL_PATH = "/tmp/sparkml_regression_pipeline"

# Clean previous save if exists
if os.path.exists(MODEL_PATH):
    shutil.rmtree(MODEL_PATH)

# Save the trained regression PipelineModel
reg_model.save(MODEL_PATH)
print(f"Pipeline saved to: {MODEL_PATH}")

# Show what was saved on disk
for root, dirs, files in os.walk(MODEL_PATH):
    level = root.replace(MODEL_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:4]:
            print(f"{indent}  {f}")

In [ ]:
from pyspark.ml import PipelineModel

# Load the pipeline back from disk
loaded_model = PipelineModel.load(MODEL_PATH)
print(f"Loaded model type: {type(loaded_model).__name__}")
print("Loaded stages:")
for i, stage in enumerate(loaded_model.stages):
    print(f"  [{i}] {type(stage).__name__}")

# Verify predictions match the original model
loaded_preds = loaded_model.transform(test_reg)

rmse_loaded = RegressionEvaluator(labelCol="tip_amount", metricName="rmse") \
              .evaluate(loaded_preds)

print(f"\nOriginal model RMSE : {rmse:.4f}")
print(f"Loaded model RMSE   : {rmse_loaded:.4f}")
print(f"Predictions match   : {abs(rmse - rmse_loaded) < 1e-6}")

---
## 9. Summary

| Slide | Concept | Demonstrated with Taxi Data |
|-------|---------|-----------------------------|
| 1 | Basic Statistics | Pearson & Spearman correlation; Summarizer; ChiSquare independence test |
| 2–3 | DataFrame / Transformer / Estimator / Pipeline / Parameter | VectorAssembler, StandardScaler, LogisticRegression |
| 4 | Linear DAG Pipeline | VectorAssembler → StandardScaler → LinearRegression → tip prediction |
| 5 | Unique stage IDs, Parameters, Persistence | `lr.uid`, `lr.setMaxIter()`, `PipelineModel.save/load` |
| 6 | CrossValidator & ParamGridBuilder | 4-combo × 3-fold CV, `parallelism=4` |

### Key rules to remember:
- `Transformer.transform()` and `Estimator.fit()` are **stateless** — always return new DataFrames
- A Pipeline **cannot reuse** the same stage object (each must have a unique UID)
- Pipelines use **schema-based runtime checking** (not compile-time)
- `CrossValidator` retrains the best model on the **full training dataset** after CV

In [ ]:
spark.stop()
print("SparkSession stopped.")